# Phase 1 - Model 5: prefix-LM (101M)

Model 4's network trained as a prefix-LM: the prompt is read bidirectionally, the continuation causally, with the loss on the continuation only.

**Run on Kaggle:** GPU T4 x2, Internet on, Hugging Face token as the secret `HF`, then Save & Run All. `SMOKE_TEST = True` (default) runs a tiny version first; set it to `False` for the real run.

## 1. Hugging Face Login

In [ ]:
import os, json, math, time, random, zipfile, heapq, copy, shutil
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from datasets import Dataset
from transformers import (PreTrainedTokenizer, DataCollatorForLanguageModeling,
                          TrainingArguments, Trainer, TrainerCallback)

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

In [ ]:
# Data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"   # any of the 4 train files can be used
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# BPE tokenizer
BPE_VOCAB_SIZE = 50000
BPE_TRAIN_CHARS = 300_000_000
# A tokenizer trained elsewhere, e.g. the mixed-domain one (add the tokenizer notebook's output as an input):
# PRETRAINED_BPE_PATH = "/kaggle/input/<tokenizer-notebook>/cs781_tokenizer_mixing/best_tokenizer/bpe_tokenizer.json"
PRETRAINED_BPE_PATH = None           # None = train the BPE tokenizer in this notebook

# Training
TRAIN_TOKENS = 220_000_000
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 1024
MODEL_CONTEXT_LENGTH = 1024
EPOCHS = 1
MAX_STEPS = -1                       # -1 = full epoch (the smoke-test benchmark tells you if it fits)
PER_DEVICE_BATCH = 8                 # blocks per GPU per forward pass
GRAD_ACCUM = 4                       # 8 x 4 x 2 GPUs = 64 blocks (~65k tokens) per optimizer step
CHUNKED_LOSS = True                  # loss computed in chunks: same value, no full fp32 (batch x 1024 x 50k) logits
CE_CHUNK_TOKENS = 2048               # tokens per loss chunk
FUSED_ADAMW = True                   # fused CUDA AdamW kernel (same update, fewer kernel launches)
LEARNING_RATE = 6e-4
WARMUP_STEPS = 200
EVAL_STEPS = 250
SAVE_STEPS = 250
LOGGING_STEPS = 25
EVAL_BLOCKS = 512                    # periodic evaluations use this many validation blocks (final one uses all)
TIME_LIMIT_HOURS = 9.0               # training stops after this; the rest of Kaggle's 12 h covers loading,
                                     # tokenization, evaluation, generation (<= 1.3 h worst case) and saving
SEED = 42
random.seed(SEED)

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1                    # 0 = greedy decoding
TOP_P = 0.95
NO_REPEAT_NGRAM_SIZE = 4             # never repeat a 4-token sequence within one continuation (0 = off)
BLOCK_SPECIAL_TOKENS = True          # never sample </s> <pad> <s> <unk>: every output reaches 128 words
GENERATION_BATCH_SIZE = 32
NUM_TEST_EXAMPLES = None             # None = whole test set
CONTINUATION_PROMPT_LEN = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS   # for the comparable "continuation loss"

# Prefix-LM: each training block = prompt (bidirectional, no loss) + continuation (causal, with loss).
# The prompt length is drawn uniformly from [PREFIX_MIN, PREFIX_MAX]; PREFIX_MAX matches the longest
# prompt used at generation time.
PREFIX_MIN = 64
PREFIX_MAX = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS

OUTPUT_DIR = "/kaggle/working/cs781_phase1"

# Real-run sizes, remembered before the smoke test shrinks them (used by the epoch-time benchmark)
FULL_BPE_VOCAB_SIZE, FULL_TRAIN_TOKENS, FULL_VAL_TOKENS, FULL_EVAL_STEPS = BPE_VOCAB_SIZE, TRAIN_TOKENS, VAL_TOKENS, EVAL_STEPS
BENCHMARK_STEPS = 20                 # timed optimizer steps of the full-size model (smoke test only)
BENCHMARK_SKIP = 3                   # first steps ignored (warm-up)
# Speed settings timed by the benchmark in 10c (smoke test only), all on the 101M shape at context 1024
# and the same ~65k tokens per optimizer step. The first row = the old settings (the 7.5k tokens/s run).
_SHAPE = dict(d_model=768, n_layers=10, n_heads=12, n_kv_heads=4, d_ff=2048, context=1024)
BENCHMARK_GRID = [
    dict(_SHAPE, chunked_loss=False, fused_adamw=False, per_device_batch=8),   # old settings
    dict(_SHAPE, chunked_loss=True,  fused_adamw=False, per_device_batch=8),   # + chunked loss
    dict(_SHAPE, chunked_loss=True,  fused_adamw=True,  per_device_batch=8),   # + fused AdamW (new default)
    dict(_SHAPE, chunked_loss=True,  fused_adamw=True,  per_device_batch=16),  # + bigger batch, GRAD_ACCUM 2
]
# Candidate TRAIN_TOKENS values checked against time, data and RAM (10c)
BENCHMARK_TOKEN_OPTIONS = [150_000_000, 200_000_000, 250_000_000, 275_000_000, 300_000_000,
                           350_000_000, 400_000_000]

# ---- SMOKE TEST: True = tiny end-to-end run (~15-25 min) that checks everything works
# ---- AND benchmarks how long one full epoch takes. Set to False for the real training run.
SMOKE_TEST = True
if SMOKE_TEST:
    BPE_VOCAB_SIZE, BPE_TRAIN_CHARS = 2000, 2_000_000
    TRAIN_TOKENS, VAL_TOKENS = 1_000_000, 100_000
    MAX_STEPS, EVAL_STEPS, SAVE_STEPS, LOGGING_STEPS, WARMUP_STEPS = 20, 10, 10, 5, 5
    NUM_TEST_EXAMPLES = 64
    OUTPUT_DIR = "/kaggle/working/cs781_phase1_smoke"   # separate folder: smoke files never mix with the real run
    print("*** SMOKE TEST MODE: tiny run, predictions are NOT meant for submission ***")

# Output paths
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-prefix-lm-101m")
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "bpe_tokenizer")
BPE_PATH = os.path.join(TOKENIZER_DIR, "bpe_tokenizer.json")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)
print("Configuration loaded. Outputs go to:", OUTPUT_DIR)

### BPE Tokenizer (implementation)

In [ ]:
_SHIFT = 20
_MASK = (1 << _SHIFT) - 1
_EOW = _MASK


class ByteLevelBPE:
    def __init__(self, vocab_size=5000):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.merges = []
        self.symbol_to_id = {}
        self._ranks = None
        self._cache = {}

    def fit(self, sentences):
        word_freq = Counter()
        for s in sentences:
            word_freq.update(s.split())

        sym = {}
        key = {}
        for b in range(256):
            i = len(self.itos)
            self.symbol_to_id[(b,)] = i
            self.itos.append([b])
            sym[i] = (b,)
            key[i] = chr(b)
        sym[_EOW] = (256,)
        key[_EOW] = chr(256)

        words, freqs = [], []
        for w, f in word_freq.items():
            words.append([b + 4 for b in w.encode("utf-8")] + [_EOW])
            freqs.append(f)
        del word_freq

        pc = defaultdict(int)
        where = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for x, y in zip(w, w[1:]):
                p = (x << _SHIFT) | y
                pc[p] += f
                where[p].add(wi)

        heap = [(-c, key[p >> _SHIFT], key[p & _MASK], p) for p, c in pc.items() if c > 1]
        heapq.heapify(heap)

        itos, merges, s2id = self.itos, self.merges, self.symbol_to_id
        target = self.target_vocab_size
        heappop, heappush = heapq.heappop, heapq.heappush

        while len(itos) < target:
            while heap:
                negc, _, _, best = heappop(heap)
                if pc.get(best, 0) == -negc:
                    break
            else:
                break
            if -negc <= 1:
                break

            a, b = best >> _SHIFT, best & _MASK
            new = len(itos)
            new_sym = sym[a] + sym[b]
            merges.append([list(sym[a]), list(sym[b])])
            s2id[new_sym] = new
            itos.append(list(new_sym))
            sym[new] = new_sym
            key[new] = key[a] + key[b]

            changed = set()
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                i = 0
                while i < len(w) - 1:
                    if w[i] == a and w[i + 1] == b:
                        if i > 0:
                            l = w[i - 1]
                            p = (l << _SHIFT) | a; pc[p] -= f; changed.add(p)
                            p = (l << _SHIFT) | new; pc[p] += f; changed.add(p); where[p].add(wi)
                        if i + 2 < len(w):
                            r = w[i + 2]
                            p = (b << _SHIFT) | r; pc[p] -= f; changed.add(p)
                            p = (new << _SHIFT) | r; pc[p] += f; changed.add(p); where[p].add(wi)
                        pc[best] -= f
                        w[i:i + 2] = [new]
                    i += 1

            pc.pop(best, None)
            changed.discard(best)
            for p in changed:
                c = pc[p]
                if c > 1:
                    heappush(heap, (-c, key[p >> _SHIFT], key[p & _MASK], p))
                elif c <= 0:
                    del pc[p]

            if len(merges) % 500 == 0:
                print(f"Merges: {len(merges):,} | Vocab: {len(itos):,} | Best pair count: {-negc:,}")

        self.vocab_size = len(itos)
        self._ranks = None
        self._cache = {}
        print(f"BPE trained: {self.vocab_size:,} vocabulary entries, {len(merges):,} merges")
        return self

    def _build_ranks(self):
        def to_id(t):
            t = tuple(t)
            return _EOW if t == (256,) else self.symbol_to_id[t]
        if not self.symbol_to_id:
            for i, s in enumerate(self.itos):
                if isinstance(s, list):
                    self.symbol_to_id[tuple(s)] = i
        self._ranks = {}
        for l, r in self.merges:
            self._ranks[(to_id(l) << _SHIFT) | to_id(r)] = self.symbol_to_id[tuple(l) + tuple(r)]

    def tokenize_word(self, word):
        out = self._cache.get(word)
        if out is not None:
            return out
        if self._ranks is None:
            self._build_ranks()
        ranks = self._ranks
        w = [b + 4 for b in word.encode("utf-8")]
        w.append(_EOW)

        while len(w) > 1:
            best_new, best_p = None, None
            for x, y in zip(w, w[1:]):
                n = ranks.get((x << _SHIFT) | y)
                if n is not None and (best_new is None or n < best_new):
                    best_new, best_p = n, (x, y)
            if best_new is None:
                break
            a, b = best_p
            i, res = 0, []
            L = len(w)
            while i < L:
                if i < L - 1 and w[i] == a and w[i + 1] == b:
                    res.append(best_new); i += 2
                else:
                    res.append(w[i]); i += 1
            w = res

        out = [1 if t == _EOW else t for t in w]
        self._cache[word] = out
        return out

    def encode_sentence(self, sentence):
        ids = []
        tw = self.tokenize_word
        for word in sentence.split():
            ids.extend(tw(word))
        return ids

    def encode_corpus(self, sentences):
        return [self.encode_sentence(s) for s in sentences]

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": self.merges, "itos": self.itos,
                       "vocab_size": self.vocab_size}, f, ensure_ascii=False)

### Hugging Face tokenizer wrapper

In [ ]:
_EOW_MARK = "▁"


def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, map(chr, cs)))


class LegalBPETokenizer(PreTrainedTokenizer):
    vocab_files_names = {"vocab_file": "bpe_tokenizer.json"}
    model_input_names = ["input_ids", "attention_mask"]

    def __init__(self, vocab_file, **kwargs):
        with open(vocab_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        bpe = ByteLevelBPE()
        bpe.itos = data["itos"]
        bpe.merges = data["merges"]
        bpe.vocab_size = data["vocab_size"]
        bpe.symbol_to_id = {}
        bpe._build_ranks()
        self.bpe = bpe
        self.eow_id = len(bpe.itos)

        byte_enc = _bytes_to_unicode()
        self.byte_decoder = {c: b for b, c in byte_enc.items()}
        self.id_to_token = list(bpe.itos[:4])
        for sym in bpe.itos[4:]:
            self.id_to_token.append("".join(_EOW_MARK if b == 256 else byte_enc[b] for b in sym))
        self.id_to_token.append(_EOW_MARK)
        self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}

        kwargs.setdefault("pad_token", "<pad>")
        kwargs.setdefault("unk_token", "<unk>")
        kwargs.setdefault("bos_token", "<s>")
        kwargs.setdefault("eos_token", "</s>")
        kwargs.setdefault("clean_up_tokenization_spaces", False)
        super().__init__(**kwargs)

    def encode_fast(self, text):
        eow, tw = self.eow_id, self.bpe.tokenize_word
        ids = []
        for word in text.split():
            ids.extend(eow if t == 1 else t for t in tw(word))
        return ids

    @property
    def vocab_size(self):
        return len(self.id_to_token)

    def get_vocab(self):
        return dict(self.token_to_id)

    def _tokenize(self, text):
        return [self.id_to_token[i] for i in self.encode_fast(text)]

    def _convert_token_to_id(self, token):
        return self.token_to_id.get(token, self.unk_token_id)

    def _convert_id_to_token(self, index):
        return self.id_to_token[index]

    def convert_tokens_to_string(self, tokens):
        out = bytearray()
        for tok in tokens:
            if tok in ("<pad>", "<unk>", "<s>", "</s>"):
                continue
            for c in tok:
                out.extend(b" " if c == _EOW_MARK else bytes([self.byte_decoder[c]]))
        return out.decode("utf-8", errors="replace").strip()

    def save_vocabulary(self, save_directory, filename_prefix=None):
        path = os.path.join(save_directory, (filename_prefix + "-" if filename_prefix else "") + "bpe_tokenizer.json")
        self.bpe.save(path)
        return (path,)

## 4. Download Training Data

In [ ]:
train_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TRAIN_FILE, repo_type="dataset", token=HF_TOKEN)
with open(train_file_path, "r", encoding="utf-8") as f:
    records = json.load(f)
print(f"Loaded {len(records):,} documents from {train_file_path}")

## 5. Train the BPE Tokenizer

In [ ]:
def bpe_training_texts(records, max_chars, seed=SEED):
    order = list(range(len(records)))
    random.Random(seed).shuffle(order)
    total = 0
    for idx in order:
        text = records[idx]["text"]
        yield text
        total += len(text)
        if total >= max_chars:
            break
    print(f"BPE training text: {total:,} characters")


if PRETRAINED_BPE_PATH:
    assert os.path.exists(PRETRAINED_BPE_PATH), f"PRETRAINED_BPE_PATH not found: {PRETRAINED_BPE_PATH}"
    shutil.copy(PRETRAINED_BPE_PATH, BPE_PATH)
    print("Using the pre-trained BPE tokenizer:", PRETRAINED_BPE_PATH)

if os.path.exists(BPE_PATH):
    print("Found existing BPE tokenizer, skipping training:", BPE_PATH)
else:
    start_time = time.time()
    bpe = ByteLevelBPE(vocab_size=BPE_VOCAB_SIZE)
    bpe.fit(bpe_training_texts(records, BPE_TRAIN_CHARS))
    bpe.save(BPE_PATH)
    print(f"Tokenizer training time: {(time.time() - start_time) / 60:.1f} min")
    print("Saved BPE tokenizer to:", BPE_PATH)

tokenizer = LegalBPETokenizer(vocab_file=BPE_PATH)
print("Vocabulary size:", len(tokenizer))

sample = " ".join(records[0]["text"].split()[:60])
sample_ids = tokenizer.encode_fast(sample)
print(f"{len(sample.split())} words -> {len(sample_ids)} tokens")
print("Round-trip OK:", tokenizer.decode(sample_ids) == sample)

## 6. Tokenize the Corpus into Train / Validation Tokens

In [ ]:
tokenize_start = time.time()
train_tokens = []
val_tokens = []

random.seed(SEED)

for data in records:
    text = data["text"]
    tokens = tokenizer.encode_fast(text) + [tokenizer.eos_token_id]

    if random.random() < 0.05:
        if len(val_tokens) < VAL_TOKENS:
            remaining = (VAL_TOKENS - len(val_tokens))
            val_tokens.extend(tokens[:remaining])
    else:
        if len(train_tokens) < TRAIN_TOKENS:
            remaining = (TRAIN_TOKENS - len(train_tokens))
            train_tokens.extend(tokens[:remaining])

    if (len(train_tokens) >= TRAIN_TOKENS and len(val_tokens) >= VAL_TOKENS):
        break

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")
tokenize_seconds = time.time() - tokenize_start
print(f"Tokenization time: {tokenize_seconds / 60:.1f} min")

## 7. Create Token Blocks

In [ ]:
def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]
    return [tokens[i:i + block_size] for i in range(0, total_length, block_size)]

train_blocks = create_blocks(train_tokens, TRAIN_BLOCK_SIZE)
val_blocks = create_blocks(val_tokens, TRAIN_BLOCK_SIZE)
print(f"Train blocks: {len(train_blocks):,} | Validation blocks: {len(val_blocks):,}")

## 8. Create Datasets

In [ ]:
train_dataset = Dataset.from_dict({"input_ids": train_blocks})
val_dataset = Dataset.from_dict({"input_ids": val_blocks})
del train_tokens, val_tokens, train_blocks, val_blocks
print(train_dataset)
print(val_dataset)

## 9. Define the Model (Prefix-LM Transformer)

In [ ]:
@dataclass
class ModernLMConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 10
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 2048                   # SwiGLU hidden size (3 matrices = params of a 3072 GELU MLP)
    max_seq_len: int = 1024
    rope_base: float = 10000.0
    norm_eps: float = 1e-5
    prefix_lm: bool = False            # True = bidirectional attention over the prompt (Option 2)


# ---- RoPE (split-half pairing, theta_i = base^(-2i/d_head)), as in A-3 ----
def build_rope_cache(max_seq_len, d_head, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin


# ---- From-scratch attention (no nn.MultiheadAttention / F.scaled_dot_product_attention) ----
# out = softmax(q k^T * scale, masked) v, computed in query blocks with the softmax statistics kept per row
# (log-sum-exp), so the full (T, S) score matrix is never stored. The backward pass is written by hand and
# recomputes each block's probabilities from the saved log-sum-exp:
#   P = exp(S - lse),  dV = P^T dO,  dP = dO V^T,  dS = P * (dP - rowsum(dO * O)),  dQ = dS K * scale,  dK = dS^T Q * scale


def _block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset):
    """Scaled, masked scores (B, H, j-i, kend) in fp32 for queries [i, j) against keys [0, kend)."""
    s = torch.matmul(q_blk, k[:, :, :kend].transpose(-2, -1)).float() * scale
    if attn_mask is not None:
        s = s.masked_fill(~attn_mask[..., i:j, :kend], float("-inf"))
    if is_causal:                                           # query row r sits at key position offset + r
        rows = torch.arange(i, j, device=s.device)[:, None] + offset
        s = s.masked_fill(torch.arange(kend, device=s.device)[None, :] > rows, float("-inf"))
    return s


class BlockwiseAttention(torch.autograd.Function):
    """q: (B, H, T, d); k, v: (B, H, S, d); attn_mask: bool, broadcastable to (B, H, T, S), True = may attend.
    Every query row must be allowed at least one key."""

    @staticmethod
    def forward(ctx, q, k, v, attn_mask, is_causal, scale, block):
        T, S = q.size(-2), k.size(-2)
        offset = S - T                                      # causal: the queries are the last T positions
        out = torch.empty_like(q)
        lse = torch.empty(q.shape[:-1], dtype=torch.float32, device=q.device)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S   # causal: later keys are masked anyway, skip them
            s = _block_scores(q[:, :, i:j], k, scale, attn_mask, is_causal, i, j, kend, offset)
            m = s.amax(-1, keepdim=True)
            p = torch.exp(s - m)
            l = p.sum(-1, keepdim=True)
            out[:, :, i:j] = torch.matmul((p / l).to(v.dtype), v[:, :, :kend])
            lse[:, :, i:j] = (m + l.log()).squeeze(-1)
        ctx.save_for_backward(q, k, v, out, lse, attn_mask)
        ctx.is_causal, ctx.scale, ctx.block = is_causal, scale, block
        return out

    @staticmethod
    def backward(ctx, d_out):
        q, k, v, out, lse, attn_mask = ctx.saved_tensors
        is_causal, scale, block = ctx.is_causal, ctx.scale, ctx.block
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        d_out = d_out.contiguous()
        dq = torch.empty_like(q)
        dk = torch.zeros(k.shape, dtype=torch.float32, device=k.device)
        dv = torch.zeros(v.shape, dtype=torch.float32, device=v.device)
        delta = (d_out.float() * out.float()).sum(-1, keepdim=True)        # rowsum(dO * O) = rowsum(P * dP)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            q_blk, do_blk = q[:, :, i:j], d_out[:, :, i:j]
            p = torch.exp(_block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset)
                          - lse[:, :, i:j, None])                           # masked entries -> exactly 0
            dv[:, :, :kend] += torch.matmul(p.to(v.dtype).transpose(-2, -1), do_blk).float()
            dp = torch.matmul(do_blk, v[:, :, :kend].transpose(-2, -1)).float()
            ds = (p * (dp - delta[:, :, i:j])).to(q.dtype)
            dq[:, :, i:j] = torch.matmul(ds, k[:, :, :kend]) * scale
            dk[:, :, :kend] += torch.matmul(ds.transpose(-2, -1), q_blk).float() * scale
        return dq, dk.to(k.dtype), dv.to(v.dtype), None, None, None, None


def scratch_attention(q, k, v, attn_mask=None, is_causal=False, scale=None, block=128):
    """Drop-in for the fused kernel: q (B, H, T, d), k/v (B, H_kv, S, d) with H_kv dividing H (GQA).
    attn_mask: bool (True = may attend), broadcastable to (B, 1, T, S)."""
    scale = q.size(-1) ** -0.5 if scale is None else scale
    B, H, T, d = q.shape
    n_rep = H // k.size(1)
    if n_rep > 1 and not is_causal:
        # GQA without copying K/V: the n_rep query heads of a group become extra query rows for that kv head
        q = q.reshape(B, k.size(1), n_rep * T, d)
        if attn_mask is not None:
            attn_mask = attn_mask.repeat(1, 1, n_rep, 1)
        out = BlockwiseAttention.apply(q, k, v, attn_mask, False, scale, block)
        return out.reshape(B, H, T, d)
    if n_rep > 1:
        k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
    return BlockwiseAttention.apply(q, k, v, attn_mask, is_causal, scale, block)


def build_attention_mask(T, key_keep=None, prefix_len=None, device=None):
    """Boolean mask (B, 1, T, T), True = may attend. None = plain causal (no mask tensor needed).
    key_keep: (B, T) True for real tokens (left padding at generation).
    prefix_len: (B,) prompt length; prompt keys are visible to every query (prefix-LM)."""
    if key_keep is None and prefix_len is None:
        return None
    i = torch.arange(T, device=device)
    allowed = (i[None, :] <= i[:, None])[None]                                 # (1, T, T) causal
    if prefix_len is not None:
        allowed = allowed | (i[None, None, :] < prefix_len[:, None, None])     # prompt is bidirectional
    if key_keep is not None:
        allowed = allowed & key_keep[:, None, :]                               # never attend to padding
        allowed = allowed | torch.eye(T, dtype=torch.bool, device=device)[None]   # pad rows see themselves (no NaN)
    return allowed[:, None]


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.to(x.dtype) * self.weight


class GQAttention(nn.Module):
    def __init__(self, cfg: ModernLMConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.n_heads, self.n_kv_heads = cfg.n_heads, cfg.n_kv_heads
        self.n_rep = cfg.n_heads // cfg.n_kv_heads
        self.d_head = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(cfg.n_heads * self.d_head, cfg.d_model, bias=False)
        cos, sin = build_rope_cache(cfg.max_seq_len, self.d_head, cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, attn_mask=None, pos_ids=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if pos_ids is None:
            cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        else:                                                   # per-row positions (left padding)
            cos, sin = self.rope_cos[pos_ids].unsqueeze(1), self.rope_sin[pos_ids].unsqueeze(1)
        q = apply_rope(q, cos.to(q.dtype), sin.to(q.dtype))
        k = apply_rope(k, cos.to(k.dtype), sin.to(k.dtype))
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        out = scratch_attention(q, k, v, attn_mask=attn_mask, is_causal=attn_mask is None)
        return self.out_proj(out.transpose(1, 2).reshape(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class ModernBlock(nn.Module):
    def __init__(self, cfg: ModernLMConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(self, x, attn_mask=None, pos_ids=None):
        x = x + self.attn(self.attn_norm(x), attn_mask, pos_ids)
        return x + self.mlp(self.mlp_norm(x))


class ModernDecoderLM(nn.Module):
    def __init__(self, cfg: ModernLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([ModernBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        # weight tying is done in forward (logits = x @ tok_emb.weight^T): no shared tensors,
        # so Trainer checkpoints save cleanly as safetensors.
        self.apply(self._init_weights)
        for block in self.blocks:                               # scaled init of residual output layers
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))
            nn.init.normal_(block.mlp.down.weight, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, attention_mask=None, prefix_len=None, last_only=False, return_hidden=False):
        B, T = idx.shape
        assert T <= self.cfg.max_seq_len, f"sequence length {T} > max_seq_len {self.cfg.max_seq_len}"
        key_keep, pos_ids = None, None
        if attention_mask is not None and not bool(attention_mask.all()):
            key_keep = attention_mask.bool()                    # only when there is padding
            pos_ids = (key_keep.long().cumsum(-1) - 1).clamp(min=0)   # positions start at the first real token
        attn_mask = build_attention_mask(T, key_keep, prefix_len if self.cfg.prefix_lm else None, idx.device)
        x = self.tok_emb(idx)
        for block in self.blocks:
            x = block(x, attn_mask, pos_ids)
        if last_only:                                           # generation: only the next-token logits
            x = x[:, -1:]
        if return_hidden:                                       # chunked loss applies the output head itself
            return self.final_norm(x)
        return F.linear(self.final_norm(x), self.tok_emb.weight)   # tied output head


import torch.utils.checkpoint


def _ce_sum(h, weight, y):
    return F.cross_entropy(F.linear(h, weight).float(), y, ignore_index=-100, reduction="sum")


def chunked_cross_entropy(hidden, weight, labels, chunk_tokens=2048):
    """Same value as cross_entropy(logits, labels) (mean over non-ignored tokens), but the logits are
    built chunk by chunk and recomputed in the backward pass, so the full (batch x T x vocab) fp32
    tensor never exists."""
    h = hidden.reshape(-1, hidden.size(-1))
    y = labels.reshape(-1)
    total = h.new_zeros((), dtype=torch.float32)
    for s in range(0, h.size(0), chunk_tokens):
        total = total + torch.utils.checkpoint.checkpoint(_ce_sum, h[s:s + chunk_tokens], weight,
                                                          y[s:s + chunk_tokens], use_reentrant=False)
    return total / (y != -100).sum().clamp(min=1)


class ModernForCausalLM(nn.Module):
    """Thin wrapper so Hugging Face Trainer can train the model. Labels are shifted here."""

    def __init__(self, cfg: ModernLMConfig, chunked_loss=False, ce_chunk_tokens=2048):
        super().__init__()
        self.lm_config = cfg
        self.lm = ModernDecoderLM(cfg)
        self.chunked_loss, self.ce_chunk_tokens = chunked_loss, ce_chunk_tokens   # speed only, not saved

    # No **kwargs on purpose: otherwise newer Trainer versions pass num_items_in_batch and
    # stop dividing the loss by gradient_accumulation_steps.
    def forward(self, input_ids, attention_mask=None, labels=None, prefix_len=None):
        if labels is None:
            return {"logits": self.lm(input_ids, attention_mask=attention_mask, prefix_len=prefix_len)}
        # training blocks are packed without padding, so the attention mask is all ones: skipping it
        # avoids a GPU -> CPU sync every forward pass
        if self.chunked_loss:
            hidden = self.lm(input_ids, prefix_len=prefix_len, return_hidden=True)
            loss = chunked_cross_entropy(hidden[:, :-1], self.lm.tok_emb.weight, labels[:, 1:], self.ce_chunk_tokens)
        else:
            logits = self.lm(input_ids, prefix_len=prefix_len)
            loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)),
                                   labels[:, 1:].reshape(-1), ignore_index=-100)
        return {"loss": loss}            # no logits: DataParallel would gather (B, T, 50k) onto GPU 0


def save_modern_model(model, out_dir):
    model = model.module if hasattr(model, "module") else model
    os.makedirs(out_dir, exist_ok=True)
    torch.save(model.lm.state_dict(), os.path.join(out_dir, "modern_model.pt"))
    with open(os.path.join(out_dir, "modern_config.json"), "w") as f:
        json.dump(asdict(model.lm_config), f, indent=2)


def load_modern_model(out_dir, map_location="cpu"):
    with open(os.path.join(out_dir, "modern_config.json")) as f:
        cfg = ModernLMConfig(**json.load(f))
    model = ModernForCausalLM(cfg)
    model.lm.load_state_dict(torch.load(os.path.join(out_dir, "modern_model.pt"), map_location=map_location))
    return model


def _sample_top_p(logits, temperature, top_p):
    if temperature <= 0:                                        # greedy decoding
        return logits.argmax(dim=-1)
    probs = torch.softmax(logits / temperature, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, dim=-1, descending=True)
    cum = torch.cumsum(sorted_probs, dim=-1)
    sorted_probs[(cum - sorted_probs) > top_p] = 0.0            # keep the smallest set with mass >= top_p
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    choice = torch.multinomial(sorted_probs, 1)
    return sorted_idx.gather(-1, choice).squeeze(-1)


def _ban_repeated_ngrams(logits, generated, n):
    """Set logits to -inf for any token that would repeat an n-gram already in that row's
    continuation. generated: list (one per row) of the token ids generated so far."""
    if n <= 0:
        return
    for r, toks in enumerate(generated):
        if len(toks) < n:
            continue
        prefix = tuple(toks[-(n - 1):])
        banned = [toks[i + n - 1] for i in range(len(toks) - n + 1) if tuple(toks[i:i + n - 1]) == prefix]
        if banned:
            logits[r, banned] = float("-inf")



class PrefixLMCollator:
    """Each block -> random prompt length p in [min_len, max_len]. Prompt tokens are visible
    bidirectionally and get no loss (label -100); the continuation gets next-token loss."""

    def __init__(self, min_len, max_len):
        self.min_len, self.max_len = min_len, max_len

    def __call__(self, features):
        input_ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
        B, T = input_ids.shape
        prefix_len = torch.randint(self.min_len, self.max_len + 1, (B,))
        labels = input_ids.clone()
        labels[torch.arange(T)[None, :] < prefix_len[:, None]] = -100
        return {"input_ids": input_ids, "labels": labels, "prefix_len": prefix_len}

### Build the model and data collator

In [ ]:
config = ModernLMConfig(
    vocab_size=len(tokenizer),
    d_model=768,
    n_layers=10,
    n_heads=12,
    n_kv_heads=4,
    d_ff=2048,
    max_seq_len=MODEL_CONTEXT_LENGTH,
    prefix_lm=True,
)
model = ModernForCausalLM(config, chunked_loss=CHUNKED_LOSS, ce_chunk_tokens=CE_CHUNK_TOKENS)
data_collator = PrefixLMCollator(PREFIX_MIN, PREFIX_MAX)
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")   # expect ~101.3M

## 10. Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=0.1,
    adam_beta1=0.9,
    adam_beta2=0.95,
    max_grad_norm=1.0,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    prediction_loss_only=True,
    seed=SEED,
    report_to="none",
    optim="adamw_torch_fused" if FUSED_ADAMW and torch.cuda.is_available() else "adamw_torch",
)
print("Blocks per optimizer step:", training_args.train_batch_size * training_args.gradient_accumulation_steps)

### 10b. Epoch-Time Benchmark (runs only when `SMOKE_TEST = True`)

In [ ]:
if SMOKE_TEST:
    class StepTimer(TrainerCallback):
        """Records a timestamp at the end of every optimizer step."""
        def __init__(self):
            self.stamps = []
        def on_step_end(self, args, state, control, **kwargs):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            self.stamps.append(time.time())

    def gpu_sync():
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    # --- full-size model: identical config, but the real 50k vocabulary (+1 end-of-word id) ---
    bench_cfg = ModernLMConfig(**{**asdict(config), "vocab_size": FULL_BPE_VOCAB_SIZE + 1})
    bench_model = ModernForCausalLM(bench_cfg, chunked_loss=CHUNKED_LOSS, ce_chunk_tokens=CE_CHUNK_TOKENS)
    print(f"Benchmark model: {sum(p.numel() for p in bench_model.parameters()) / 1e6:.2f}M parameters")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(i)

    # --- 1) training speed with the real training arguments ---
    bench_args = copy.deepcopy(training_args)
    bench_args.output_dir = os.path.join(OUTPUT_DIR, "benchmark")
    bench_args.max_steps = BENCHMARK_STEPS
    bench_args.warmup_steps = 0
    bench_args.eval_strategy = "no"
    bench_args.save_strategy = "no"
    bench_args.logging_steps = 5
    timer = StepTimer()
    bench_trainer = Trainer(model=bench_model, args=bench_args, train_dataset=train_dataset,
                            data_collator=data_collator, processing_class=tokenizer, callbacks=[timer])
    bench_trainer.train()
    step_times = sorted(b - a for a, b in zip(timer.stamps[BENCHMARK_SKIP:], timer.stamps[BENCHMARK_SKIP + 1:]))
    sec_per_step = step_times[len(step_times) // 2]

    # --- 2) evaluation speed ---
    n_eval_bench = min(64, len(val_dataset))
    gpu_sync(); t0 = time.time()
    bench_trainer.evaluate(eval_dataset=val_dataset.select(range(n_eval_bench)))
    gpu_sync()
    sec_per_block_eval = (time.time() - t0) / n_eval_bench
    sec_per_eval = sec_per_block_eval * EVAL_BLOCKS

    # --- 3) generation speed: one full batch, all MAX_GENERATION_TOKENS steps (worst case) ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    m = bench_model.to(device).eval()
    B = GENERATION_BATCH_SIZE
    L = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
    ids = torch.randint(4, len(tokenizer), (B, L), device=device)
    plen = torch.full((B,), L, device=device)
    generated = [[] for _ in range(B)]
    gpu_sync(); t0 = time.time()
    with torch.inference_mode():
        for _ in range(MAX_GENERATION_TOKENS):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = m.lm(ids, prefix_len=plen, last_only=True)[:, -1, :]
            logits = logits.float()
            _ban_repeated_ngrams(logits, generated, NO_REPEAT_NGRAM_SIZE)
            nxt = _sample_top_p(logits, TEMPERATURE, TOP_P)
            ids = torch.cat([ids, nxt[:, None]], dim=1)
            for row, t in zip(generated, nxt.tolist()):
                row.append(t)
    gpu_sync()
    sec_per_gen_batch = time.time() - t0

    test_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
    with open(test_path, "r", encoding="utf-8") as f:
        n_test = len(json.load(f))

    # --- estimates for the REAL run ---
    blocks_per_step = bench_args.train_batch_size * bench_args.gradient_accumulation_steps
    full_blocks = FULL_TRAIN_TOKENS // TRAIN_BLOCK_SIZE
    steps_per_epoch = math.ceil(math.ceil(full_blocks / bench_args.train_batch_size) / bench_args.gradient_accumulation_steps)
    epoch_hours = sec_per_step * steps_per_epoch / 3600
    n_evals = steps_per_epoch // FULL_EVAL_STEPS
    full_val_blocks = FULL_VAL_TOKENS // TRAIN_BLOCK_SIZE
    final_eval_blocks = full_val_blocks + min(1024, full_val_blocks)              # final evaluate() + continuation loss
    eval_hours = (n_evals * sec_per_eval + final_eval_blocks * sec_per_block_eval) / 3600
    gen_hours = math.ceil(n_test / B) * sec_per_gen_batch / 3600

    sec_per_step_with_eval = sec_per_step + sec_per_eval / FULL_EVAL_STEPS     # TimeLimit counts eval time too
    steps_in_budget = int(TIME_LIMIT_HOURS * 3600 / sec_per_step_with_eval)

    print("\n" + "=" * 66)
    print("EPOCH-TIME BENCHMARK (estimates for SMOKE_TEST = False)")
    print("=" * 66)
    print(f"GPUs used:                     {max(1, torch.cuda.device_count())}")
    print(f"Blocks per optimizer step:     {blocks_per_step}  ({blocks_per_step * TRAIN_BLOCK_SIZE:,} tokens)")
    print(f"Seconds per optimizer step:    {sec_per_step:.2f} s   (median of {len(step_times)} steps)")
    print(f"Training throughput:           {blocks_per_step * TRAIN_BLOCK_SIZE / sec_per_step:,.0f} tokens/s")
    print(f"Optimizer steps per epoch:     {steps_per_epoch:,}   ({full_blocks:,} blocks of {TRAIN_BLOCK_SIZE})")
    print(f"ONE EPOCH OF TRAINING:         {epoch_hours:.2f} h")
    print(f"Evaluations:                   {eval_hours:.2f} h")
    print(f"Test generation ({n_test} prompts): {gen_hours:.2f} h (worst case)")
    print(f"Training + eval + generation:  {epoch_hours + eval_hours + gen_hours:.2f} h  (+ BPE training and tokenization on CPU)")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            used = torch.cuda.max_memory_allocated(i) / 2**30
            total = torch.cuda.get_device_properties(i).total_memory / 2**30
            print(f"Peak memory GPU {i}:             {used:.1f} / {total:.1f} GiB")
    print("-" * 66)
    if steps_in_budget >= steps_per_epoch:
        print(f"A full epoch fits in TIME_LIMIT_HOURS = {TIME_LIMIT_HOURS}. Keep MAX_STEPS = -1.")
        budget_tokens = steps_in_budget * blocks_per_step * TRAIN_BLOCK_SIZE
        print(f"The budget allows ~{budget_tokens / 1e6:,.0f}M training tokens: you can raise TRAIN_TOKENS up to that")
        print("(the train file must contain that many tokens; the tokenize cell prints how many it found).")
    else:
        print(f"A full epoch does NOT fit in TIME_LIMIT_HOURS = {TIME_LIMIT_HOURS} h.")
        print(f"Set MAX_STEPS = {steps_in_budget} for the real run (~{100 * steps_in_budget / steps_per_epoch:.0f}% of an epoch),")
        print("so the cosine learning-rate schedule completes instead of being cut off by the time limit.")
    print("=" * 66)

    del bench_trainer, bench_model, m, ids, logits
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### 10c. Speed Settings and Token-Budget Benchmark (runs only when `SMOKE_TEST = True`)

In [ ]:
if SMOKE_TEST:
    import pandas as pd
    n_gpus = max(1, torch.cuda.device_count())
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokens_per_step = PER_DEVICE_BATCH * n_gpus * GRAD_ACCUM * TRAIN_BLOCK_SIZE
    n_gen_batches = math.ceil(n_test / GENERATION_BATCH_SIZE)       # n_test: from the 10b benchmark
    GEN_TIMED_STEPS = 32
    grid_rows = []

    for g in BENCHMARK_GRID:
        ctx = g["context"]
        ck, fused = g.get("chunked_loss", CHUNKED_LOSS), g.get("fused_adamw", FUSED_ADAMW)
        per_dev = g.get("per_device_batch", max(1, PER_DEVICE_BATCH * TRAIN_BLOCK_SIZE // ctx))
        accum = max(1, tokens_per_step // (per_dev * n_gpus * ctx))     # same tokens per optimizer step for every row
        row = {"config": f'{"chunked" if ck else "full"} loss, {"fused" if fused else "plain"} AdamW, batch {per_dev}x{accum}'}
        row.update(chunked_loss=ck, fused_adamw=fused, per_device_batch=per_dev, grad_accum=accum)
        tr = bm = m = ds = ids = logits = None
        try:
            cfg = ModernLMConfig(vocab_size=FULL_BPE_VOCAB_SIZE + 1, d_model=g["d_model"], n_layers=g["n_layers"],
                                 n_heads=g["n_heads"], n_kv_heads=g["n_kv_heads"], d_ff=g["d_ff"],
                                 max_seq_len=ctx, prefix_lm=config.prefix_lm)
            bm = ModernForCausalLM(cfg, chunked_loss=ck, ce_chunk_tokens=CE_CHUNK_TOKENS)
            row["params_M"] = sum(p.numel() for p in bm.parameters()) / 1e6
            n_blocks = (BENCHMARK_STEPS + 1) * per_dev * n_gpus * accum
            ds = Dataset.from_dict({"input_ids": torch.randint(4, FULL_BPE_VOCAB_SIZE, (n_blocks, ctx)).tolist()})
            collator = PrefixLMCollator(PREFIX_MIN, ctx - MAX_GENERATION_TOKENS) if config.prefix_lm else data_collator
            if torch.cuda.is_available():
                for i in range(torch.cuda.device_count()):
                    torch.cuda.reset_peak_memory_stats(i)

            # training speed
            args = copy.deepcopy(training_args)
            args.output_dir = os.path.join(OUTPUT_DIR, "benchmark_grid")
            args.max_steps = BENCHMARK_STEPS
            args.per_device_train_batch_size = args.per_device_eval_batch_size = per_dev
            args.gradient_accumulation_steps = accum
            args.optim = "adamw_torch_fused" if fused and torch.cuda.is_available() else "adamw_torch"
            args.warmup_steps = 0
            args.eval_strategy = "no"
            args.save_strategy = "no"
            args.logging_steps = BENCHMARK_STEPS
            timer = StepTimer()
            tr = Trainer(model=bm, args=args, train_dataset=ds, data_collator=collator,
                         processing_class=tokenizer, callbacks=[timer])
            tr.train()
            st = sorted(b - a for a, b in zip(timer.stamps[BENCHMARK_SKIP:], timer.stamps[BENCHMARK_SKIP + 1:]))
            sec_per_step = st[len(st) // 2]

            # evaluation speed
            n_ev = min(32, len(ds))
            gpu_sync(); t0 = time.time()
            tr.evaluate(eval_dataset=ds.select(range(n_ev)))
            gpu_sync()
            sec_per_eval_token = (time.time() - t0) / (n_ev * ctx)

            # generation speed: GEN_TIMED_STEPS steps in the middle of the generation window
            # (no KV cache, so a step costs more as the sequence grows; the middle is the average)
            m = bm.to(device).eval()
            B, L0 = GENERATION_BATCH_SIZE, ctx - MAX_GENERATION_TOKENS // 2
            ids = torch.randint(4, FULL_BPE_VOCAB_SIZE, (B, L0), device=device)
            plen = torch.full((B,), ctx - MAX_GENERATION_TOKENS, device=device)
            generated = [[] for _ in range(B)]
            gpu_sync(); t0 = time.time()
            with torch.inference_mode():
                for _ in range(GEN_TIMED_STEPS):
                    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                        logits = m.lm(ids, prefix_len=plen, last_only=True)[:, -1, :].float()
                    _ban_repeated_ngrams(logits, generated, NO_REPEAT_NGRAM_SIZE)
                    nxt = _sample_top_p(logits, TEMPERATURE, TOP_P)
                    ids = torch.cat([ids[:, 1:], nxt[:, None]], dim=1)       # keep the length at L0
                    for r_, t in zip(generated, nxt.tolist()):
                        r_.append(t)
            gpu_sync()
            gen_sec_per_batch = (time.time() - t0) * MAX_GENERATION_TOKENS / GEN_TIMED_STEPS

            # estimates for the real run
            tok_s = tokens_per_step / sec_per_step
            eval_sec_per_train_token = EVAL_BLOCKS * TRAIN_BLOCK_SIZE * sec_per_eval_token / (FULL_EVAL_STEPS * tokens_per_step)
            sec_per_token = 1 / tok_s + eval_sec_per_train_token
            row["sec_per_step"] = sec_per_step
            row["tokens_per_sec"] = tok_s
            if torch.cuda.is_available():
                row["peak_mem_GB"] = max(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())) / 2**30
            row["sec_per_token"] = sec_per_token
            row["max_tokens_M"] = TIME_LIMIT_HOURS * 3600 / sec_per_token / 1e6
            row["speedup"] = tok_s / grid_rows[0]["tokens_per_sec"] if grid_rows and "tokens_per_sec" in grid_rows[0] else 1.0
            row["gen_h"] = n_gen_batches * gen_sec_per_batch / 3600
            print(f"{row['config']:42s} {sec_per_step:5.2f} s/step | {tok_s:9,.0f} tok/s (x{row['speedup']:.2f}) | "
                  f"peak {row.get('peak_mem_GB', float('nan')):4.1f} GB | max {row['max_tokens_M']:6,.0f}M tokens in {TIME_LIMIT_HOURS} h")
        except Exception as e:                              # usually CUDA out of memory
            row["error"] = "OOM" if "out of memory" in str(e).lower() else f"{type(e).__name__}: {str(e)[:100]}"
            print(f"{row['config']:42s} FAILED: {row['error']}")
        grid_rows.append(row)
        tr = bm = m = ds = ids = logits = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    grid_df = pd.DataFrame(grid_rows)
    grid_df.to_csv(os.path.join(OUTPUT_DIR, "benchmark_grid.csv"), index=False)
    pd.set_option("display.width", 250)
    print("\n" + "=" * 100)
    print(f"TABLE 1: SPEED SETTINGS, 101M MODEL, CONTEXT {TRAIN_BLOCK_SIZE}  ({tokens_per_step:,} tokens per optimizer step, "
          f"{n_gpus} GPU(s), TIME_LIMIT_HOURS = {TIME_LIMIT_HOURS} h)")
    print("=" * 100)
    print(grid_df.drop(columns=["sec_per_token"], errors="ignore").round(2).to_string(index=False))

    # ---- table 2: candidate token budgets ----
    # data: ~4.5 characters per token with a 50k tokenizer (tokenizer study)
    train_chars = sum(len(r["text"]) for r in records if isinstance(r.get("text"), str))
    file_tokens = train_chars / 4.5
    # tokenization speed of this run (single process; approximate for the 50k tokenizer)
    tokenize_tok_per_s = (len(train_dataset) + len(val_dataset)) * TRAIN_BLOCK_SIZE / max(tokenize_seconds, 1e-9)
    # RAM: Python list of tokens + its truncated copy + the list of blocks (~8 bytes per token each),
    # on top of what the process already uses (the loaded train file, the model, ...)
    try:
        import psutil
        ram_now = psutil.Process().memory_info().rss / 2**30
        ram_total = psutil.virtual_memory().total / 2**30
    except Exception:
        ram_now, ram_total = float("nan"), float("nan")

    ok_rows = [r for r in grid_rows if "error" not in r]
    budget_rows = []
    for T in BENCHMARK_TOKEN_OPTIONS:
        b = {"train_tokens_M": T / 1e6, "passes_over_file": T / file_tokens,
             "tokenize_h": T / tokenize_tok_per_s / 3600, "ram_GB": ram_now + 24 * T / 2**30}
        for r in ok_rows:
            h = T * r["sec_per_token"] / 3600
            b[r["config"]] = f"{h:5.2f} h " + ("ok" if h <= TIME_LIMIT_HOURS else "> limit")
        budget_rows.append(b)
    budget_df = pd.DataFrame(budget_rows)
    budget_df.to_csv(os.path.join(OUTPUT_DIR, "benchmark_token_budget.csv"), index=False)
    print("\n" + "=" * 100)
    print(f"TABLE 2: CANDIDATE TRAIN_TOKENS  (time limit {TIME_LIMIT_HOURS} h | {TRAIN_FILE.split('/')[-1]} holds "
          f"~{file_tokens / 1e6:,.0f}M tokens | RAM {ram_total:.0f} GB)")
    print("=" * 100)
    print(budget_df.round(2).to_string(index=False))

    print("\nLargest TRAIN_TOKENS that passes every check (time limit, one pass over the file, RAM < 85%):")
    for r in ok_rows:
        fits = [T for T in BENCHMARK_TOKEN_OPTIONS
                if T * r["sec_per_token"] / 3600 <= TIME_LIMIT_HOURS and T <= file_tokens
                and not (ram_now + 24 * T / 2**30 > 0.85 * ram_total)]
        time_only = [T for T in BENCHMARK_TOKEN_OPTIONS if T * r["sec_per_token"] / 3600 <= TIME_LIMIT_HOURS]
        print(f"  {r['config']:42s} {max(fits) / 1e6:>7,.0f}M" if fits else f"  {r['config']:42s}    none",
              f"| time alone allows up to ~{r['max_tokens_M']:,.0f}M"
              + (f" (largest option: {max(time_only) / 1e6:,.0f}M)" if time_only else ""))
    ok_rows_fast = sorted(ok_rows, key=lambda r: -r["tokens_per_sec"])
    if ok_rows_fast:
        f_ = ok_rows_fast[0]
        print(f"\nFastest setting: {f_['config']}  ->  CHUNKED_LOSS = {f_['chunked_loss']}, FUSED_ADAMW = {f_['fused_adamw']}, "
              f"PER_DEVICE_BATCH = {f_['per_device_batch']}, GRAD_ACCUM = {f_['grad_accum']}")
    print("Budgets above the file size need more than one epoch over it, or extra training files.")
    print("Remember the real run also spends time on BPE training, tokenization (tokenize_h) and generation (gen_h).")
    print("Saved:", os.path.join(OUTPUT_DIR, "benchmark_grid.csv"), "and", os.path.join(OUTPUT_DIR, "benchmark_token_budget.csv"))

## 11. Training

In [ ]:
class TimeLimit(TrainerCallback):
    """Safety net: stop training after a fixed wall-clock budget."""
    def __init__(self, seconds):
        self.seconds = seconds
    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.t0 > self.seconds:
            control.should_training_stop = True


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset.select(range(min(EVAL_BLOCKS, len(val_dataset)))),
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[TimeLimit(TIME_LIMIT_HOURS * 3600)],
)

start_time = time.time()
trainer.train()
training_time = time.time() - start_time

tokens_seen = trainer.state.global_step * training_args.train_batch_size * training_args.gradient_accumulation_steps * TRAIN_BLOCK_SIZE
print(f"Steps: {trainer.state.global_step:,} | Training time: {training_time / 3600:.2f} hours")
print(f"Tokens seen: {tokens_seen:,} ({tokens_seen / training_time:,.0f} tokens/sec)")

## 12. Perplexity

In [ ]:
def continuation_loss(model, dataset, prompt_len, n_blocks=1024, batch_size=8):
    """Mean loss on the tokens after the first prompt_len tokens of each block (comparable across models)."""
    m = model.module if hasattr(model, "module") else model
    dev = next(m.parameters()).device
    m.eval()
    total, count = 0.0, 0
    with torch.inference_mode(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
        for s in range(0, min(n_blocks, len(dataset)), batch_size):
            ids = torch.tensor(dataset[s:s + batch_size]["input_ids"], dtype=torch.long, device=dev)
            labels = ids.clone()
            labels[:, :prompt_len] = -100
            plen = torch.full((ids.size(0),), prompt_len, device=dev)
            logits = m.lm(ids, prefix_len=plen)
            loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)),
                                   labels[:, 1:].reshape(-1), ignore_index=-100, reduction="sum")
            total += loss.item()
            count += (labels[:, 1:] != -100).sum().item()
    return total / count


eval_loss = trainer.evaluate(eval_dataset=val_dataset)["eval_loss"]
print(f"Validation loss: {eval_loss:.6f} | perplexity: {math.exp(eval_loss):.4f}")
cont_loss = continuation_loss(trainer.model, val_dataset, CONTINUATION_PROMPT_LEN)
print(f"Continuation loss (after a {CONTINUATION_PROMPT_LEN}-token prompt): {cont_loss:.6f} | perplexity: {math.exp(cont_loss):.4f}")

## 13. Plot Training and Validation Loss

In [ ]:
logs = trainer.state.log_history
train_log = [(l["step"], l["loss"]) for l in logs if "loss" in l]
eval_log = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]

plt.figure(figsize=(10, 5))
if train_log:
    plt.plot(*zip(*train_log), label="Training Loss")
if eval_log:
    plt.plot(*zip(*eval_log), label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Phase 1 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()
print("Loss plot saved to:", LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
save_modern_model(trainer.model, MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print("Final model saved to:", MODEL_OUTPUT_DIR)

## 15. Push Model to Hugging Face Hub

In [ ]:
# HF_MODEL_REPO = "your-username/your-model-name"
# from huggingface_hub import HfApi
# HfApi().upload_folder(folder_path=MODEL_OUTPUT_DIR, repo_id=HF_MODEL_REPO, token=HF_TOKEN, create_repo=True)
# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from Hugging Face

In [ ]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
if NUM_TEST_EXAMPLES:
    test_data = dict(list(test_data.items())[:NUM_TEST_EXAMPLES])
print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
generation_tokenizer = LegalBPETokenizer.from_pretrained(MODEL_OUTPUT_DIR)
generation_model = load_modern_model(MODEL_OUTPUT_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device).eval()
print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")

## 19. Generate 128 Words (words, not tokens)

In [ ]:
def generate_batch(prompts, model, tokenizer, device, target_words=128, check_every=16):
    max_prompt = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
    eos_id, pad_id = tokenizer.eos_token_id, tokenizer.pad_token_id
    seqs = []
    for p in prompts:
        text = p if isinstance(p, str) else ("" if p is None else str(p))
        ids = tokenizer.encode_fast(text)[-max_prompt:]
        seqs.append(ids if ids else [eos_id])      # empty prompt -> start like a new document
    L = max(len(s) for s in seqs)
    input_ids = torch.tensor([[pad_id] * (L - len(s)) + s for s in seqs], dtype=torch.long, device=device)
    attention_mask = torch.tensor([[0] * (L - len(s)) + [1] * len(s) for s in seqs], dtype=torch.long, device=device)
    prompt_len = input_ids.size(1)
    prefix_len = torch.full((len(seqs),), prompt_len, dtype=torch.long, device=device)   # used by the prefix-LM only
    banned_ids = [pad_id, tokenizer.unk_token_id, tokenizer.bos_token_id, eos_id] if BLOCK_SPECIAL_TOKENS else []
    finished = torch.zeros(len(seqs), dtype=torch.bool, device=device)
    generated = [[] for _ in seqs]
    use_amp = device.type == "cuda"

    with torch.inference_mode():
        for step in range(MAX_GENERATION_TOKENS):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model.lm(input_ids, attention_mask=attention_mask, prefix_len=prefix_len, last_only=True)[:, -1, :]
            logits = logits.float()
            if banned_ids:
                logits[:, banned_ids] = float("-inf")
            _ban_repeated_ngrams(logits, generated, NO_REPEAT_NGRAM_SIZE)
            next_tok = _sample_top_p(logits, TEMPERATURE, TOP_P)
            next_tok = torch.where(finished, torch.full_like(next_tok, eos_id), next_tok)
            input_ids = torch.cat([input_ids, next_tok[:, None]], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_tok[:, None])], dim=1)
            finished |= next_tok == eos_id
            for row, t in zip(generated, next_tok.tolist()):
                row.append(t)

            if finished.all():
                break
            if (step + 1) % check_every == 0:
                texts = tokenizer.batch_decode(input_ids[:, prompt_len:], skip_special_tokens=True)
                if all(f or len(t.split()) > target_words for f, t in zip(finished.tolist(), texts)):
                    break

    predictions = []
    for i in range(len(prompts)):
        generated_text = tokenizer.decode(input_ids[i, prompt_len:], skip_special_tokens=True)
        predictions.append(" ".join(generated_text.split()[:target_words]))
    return predictions

## 20. Generate Predictions for the Test Set

In [ ]:
predictions = {}
items = list(test_data.items())
total_examples = len(items)

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_predictions = generate_batch(
        prompts=[record["text"] for _, record in batch],
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=TARGET_GENERATION_WORDS,
    )
    for (example_id, _), prediction in zip(batch, batch_predictions):
        predictions[str(example_id)] = {"text": prediction}

    completed = start + len(batch)
    print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

## 21. Save prediction.json

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print("prediction.json saved:", PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")
print("Codabench submission created:", SUBMISSION_ZIP)

## 23. Final Checks

In [ ]:
assert math.isfinite(eval_loss) and math.isfinite(cont_loss), "loss is NaN/inf"
for fname in ("modern_model.pt", "modern_config.json", "bpe_tokenizer.json"):
    assert os.path.exists(os.path.join(MODEL_OUTPUT_DIR, fname)), f"missing {fname}"
with open(PREDICTION_PATH, encoding="utf-8") as f:
    saved = json.load(f)
assert set(saved) == {str(k) for k in test_data}, "prediction ids do not match the test ids"
lengths = [len(v["text"].split()) for v in saved.values()]
print(f"Prediction length (words): min {min(lengths)}, max {max(lengths)}, mean {sum(lengths) / len(lengths):.1f}")
assert min(lengths) > 0, "some predictions are empty"
short = sum(n < TARGET_GENERATION_WORDS for n in lengths)
if short:   # not fatal: a barely-trained model can need more than MAX_GENERATION_TOKENS tokens for 128 words
    print(f"WARNING: {short}/{len(lengths)} predictions are shorter than {TARGET_GENERATION_WORDS} words "
          "(raise MAX_GENERATION_TOKENS if this happens in the real run)")
with zipfile.ZipFile(SUBMISSION_ZIP) as z:
    assert z.namelist() == ["prediction.json"], z.namelist()
print("\nSample prediction:", next(iter(saved.values()))["text"][:300], "...")

print("\nOutput folder:", OUTPUT_DIR)
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs[:] = [d for d in dirs if not d.startswith(("checkpoint-", "benchmark"))]
    for fn in files:
        path = os.path.join(root, fn)
        print(f"  {os.path.relpath(path, OUTPUT_DIR):55s} {os.path.getsize(path) / 2**20:9.1f} MB")
print("\n" + ("SMOKE TEST PASSED: set SMOKE_TEST = False and Save & Run All again." if SMOKE_TEST
              else "ALL CHECKS PASSED: download submission.zip from the Output tab."))